# Neon glow for line and flow glyphs

A soft glow under a line makes it pop on a dark background — the "cyberpunk" look. cleopatra implements it
natively (no extra dependency) as [`add_line_glow`](../../../src/cleopatra/colors.py) and exposes it as an
opt-in `glow=` option on [`LineGlyph`](../../../src/cleopatra/line_glyph.py) and
[`FlowGlyph`](../../../src/cleopatra/flow_glyph.py). This notebook covers:

- `glow=True` (and a tuning dict) on `LineGlyph`;
- the `add_line_glow` primitive applied to any matplotlib lines;
- `glow=` on `FlowGlyph`.

The technique: redraw each line several times at growing width and low opacity, so the overlaps blur into a halo.

## Setup

We use a real signal: the area-mean of the 45 daily European temperature maps — a smooth early-summer warming curve.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from cleopatra.glyphs.primitives.line_glyph import LineGlyph
from cleopatra.glyphs.primitives.flow_glyph import FlowGlyph
from cleopatra.styling.colors import add_line_glow

DATA = Path("../../../examples/data")
t2m = np.load(DATA / "europe_t2m.npz")
daily_mean = np.nanmean(t2m["celsius"], axis=(1, 2))   # one value per day
day = np.arange(daily_mean.size)
daily_mean.shape

## A plain line first

Here is the series with no glow, for reference.

In [ ]:
glyph = LineGlyph(day, daily_mean)
fig, ax, lines = glyph.line()
ax.set_title("Europe mean 2 m temperature — plain")
ax.set_xlabel("day")
ax.set_ylabel("°C")

## Turn on the glow

Add `glow=True`. The data line is unchanged; a halo is drawn beneath it.

In [ ]:
glyph = LineGlyph(day, daily_mean, glow=True)
fig, ax, lines = glyph.line(color="deepskyblue")
ax.set_facecolor("#10141f")
fig.set_facecolor("#10141f")
ax.set_title("With glow", color="white")
ax.tick_params(colors="white")

The glow reads best on a dark background, where the low-opacity copies accumulate into a soft light.

## Tuning the glow

Pass a dict instead of `True` to tune it:

| key | meaning | typical |
|-----|---------|---------|
| `n_glow` | number of halo copies | 6–10 |
| `alpha` | opacity of **each** copy | 0.03–0.08 |
| `linewidth_step` | width added per copy (points) | 1–2 |

More copies and a larger step give a wider, softer halo.

In [ ]:
glyph = LineGlyph(day, daily_mean, glow={"n_glow": 10, "alpha": 0.07, "linewidth_step": 2.0})
fig, ax, lines = glyph.line(color="magenta")
ax.set_facecolor("#10141f")
fig.set_facecolor("#10141f")
ax.set_title("Wider halo (n_glow=10)", color="white")
ax.tick_params(colors="white")

## The `add_line_glow` primitive

The glow is available as a standalone function that haloes any existing matplotlib lines — useful for multi-series plots you build yourself. Here we overlay three smoothed views of the same series and glow them all at once.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.set_facecolor("#10141f")
fig.set_facecolor("#10141f")
for window, colour in [(1, "#00e5ff"), (3, "#ff4dd2"), (7, "#b6ff00")]:
    smooth = np.convolve(daily_mean, np.ones(window) / window, mode="same")
    ax.plot(day, smooth, color=colour, linewidth=2)
add_line_glow(ax, n_glow=8, alpha=0.05, linewidth_step=1.5)
ax.set_title("add_line_glow on three series", color="white")
ax.tick_params(colors="white")

Each line keeps its own colour in the halo — `add_line_glow` reads the colour of every line it haloes.

## Glow on a FlowGlyph

The same `glow=` option works on `FlowGlyph`, whose polylines can be width-scaled by a magnitude. Below are a few illustrative flow segments with a magnitude each; the glow traces the width-scaled lines.

In [ ]:
paths = [
    np.array([[0.0, 0.0], [1.0, 0.6], [2.0, 0.5], [3.0, 1.1]]),
    np.array([[0.0, 1.4], [1.2, 1.0], [2.1, 1.3], [3.0, 0.9]]),
    np.array([[0.2, 0.4], [1.1, 1.6], [2.3, 1.9]]),
]
magnitudes = np.array([3.0, 1.5, 2.2])
glyph = FlowGlyph(paths, values=magnitudes, width_limits=(2, 9), glow=True)
fig, ax, lc = glyph.plot()
ax.set_facecolor("#10141f")
fig.set_facecolor("#10141f")
ax.set_title("FlowGlyph with glow", color="white")
ax.tick_params(colors="white")

## Takeaway

- `glow=True` (or a `{n_glow, alpha, linewidth_step}` dict) adds a neon halo to `LineGlyph` and `FlowGlyph`.
- [`add_line_glow`](../../../src/cleopatra/colors.py) haloes any matplotlib lines directly, per-line colour.
- Glow reads best on a dark background; keep `alpha` low so the copies accumulate softly.